<a href="https://colab.research.google.com/github/Randasabag/Projet-5-ML/blob/main/P5_USE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import tensorflow as tf
import tensorflow_hub as hub

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
from scipy.stats import shapiro
import statsmodels.api as sm
import statsmodels.formula.api as smf
from datetime import datetime,date,timedelta
import datetime as dt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
df_net=pd.read_csv('/content/drive/MyDrive/MACHINE_LEARNING/df_net.csv', sep=',')
df_net

,LastActivityDate,Title,Tags,Score,sentence_bow,sentence_bow_lem,sentence_dl
0,2022-08-29 20:14:36,Determine the type of an object?,<python><dictionary><types><typeof>,2164,determine type object,determine type object,determine the type of an object ?
1,2022-07-02 05:29:08,Why can't Python parse this JSON data?,<python><json><parsing>,1503,why python parse json data,why python parse json data,why can t python parse this json data ?
2,2023-11-20 22:23:13,Best way to convert string to bytes in Python 3?,<python><string><character-encoding><python-3.x>,1482,best way convert string bytes python,best way convert string byte python,best way to convert string to bytes in python 3 ?
3,2022-11-09 15:44:56,Display number with leading zeros,<python><integer><string-formatting>,1419,display number leading zeros,display number leading zero,display number with leading zeros
4,2023-07-28 17:04:02,Create a Pandas Dataframe by appending one row...,<python><pandas><dataframe><append>,1399,create pandas dataframe appending one row time,create panda dataframe appending one row time,create a pandas dataframe by appending one row...
5,2022-11-23 08:30:50,Should I use 'has_key()' or 'in' on Python dicts?,<python><dictionary>,1122,should use key python dicts,should use key python dicts,should i use has key ( ) or in on python dicts ?
6,2022-08-15 07:57:04,How to check if type of a variable is string?,<python><string><variables><types>,1051,how check type variable string,how check type variable string,how to check if type of a variable is string ?
7,2023-01-27 10:02:02,Which exception should I raise on bad/illegal ...,<python><exception><arguments>,820,which exception raise bad illegal argument com...,which exception raise bad illegal argument com...,which exception should i raise on bad illegal ...
8,2022-05-31 17:20:40,Using @property versus getters and setters,<python><properties><getter-setter>,798,using property versus getters setters,using property versus getters setter,using property versus getters and setters
9,2023-01-29 17:03:17,"How can I check if a string represents an int,...",<python><string><integer>,733,how check string represents int without using ...,how check string represents int without using ...,how can i check if a string represents an int ...


In [24]:
# Load pre-trained universal sentence encoder model
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

In [25]:
# Sentences for which you want to create embeddings,
# passed as an array in embed()
list_sentences = df_net['sentence_dl'].tolist()
print(list_sentences)
embeddings = embed(df_net['sentence_dl'])
# Convertir les embeddings en numpy array pour un traitement ultérieur
embeddings_np = embeddings.numpy()

# Printing embeddings of each sentence
print(embeddings_np)

['determine the type of an object ?', 'why can t python parse this json data ?', 'best way to convert string to bytes in python 3 ?', 'display number with leading zeros', 'create a pandas dataframe by appending one row at a time', 'should i use has key ( ) or in on python dicts ?', 'how to check if type of a variable is string ?', 'which exception should i raise on bad illegal argument combinations in python ?', 'using property versus getters and setters', 'how can i check if a string represents an int , without using try except ?', 'how can i use pickle to save a dict ( or any other python object ) ?', 'getting a map ( ) to return a list in python 3.x', 'how can i pivot a dataframe ?', 'determine if variable is defined in python', 'reading json from a file', 'python dictionary comprehension', "is there a way to perform `` if '' in python s lambda ?", 'convert a list of characters into a string', 'how to add to the pythonpath in windows , so it finds my modules packages ?', 'how to fin

In [26]:
# To print each embeddings along with its corresponding
# sentence below code can be used.
for i in range(len(list_sentences)):
    print(list_sentences[i])
    print(embeddings[i])

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
 -6.58627674e-02  1.37295090e-02  8.47445428e-02  4.46116589e-02
  7.92699456e-02 -3.41255181e-02  3.91644426e-02  1.47339981e-02
 -1.00537911e-02 -5.93076274e-02  8.08746144e-02 -3.22772041e-02
 -1.09998221e-02  6.01404645e-02  4.96639684e-02  3.42592821e-02
  1.85573883e-02  7.81171545e-02  5.49696833e-02  3.50731388e-02
  3.70104983e-02  1.47254476e-02  2.32921522e-02 -7.36271217e-02
 -2.48164628e-02 -8.22465420e-02 -5.60705326e-02  1.18168816e-03
 -1.34689454e-03 -9.42719262e-03 -9.74360015e-03  3.38166915e-02
  1.48446122e-02 -3.72792892e-02 -1.32196757e-03 -6.85120188e-03
 -5.16786613e-02  1.55117391e-02  1.85492225e-02 -5.05275838e-02
 -4.90088798e-02 -8.36824700e-02  1.22648161e-02 -3.53459939e-02
 -2.81941406e-02 -7.48446658e-02  7.33984858e-02 -5.17900027e-02
 -6.03617355e-02 -5.92241734e-02  3.60750221e-02 -5.44160865e-02
 -2.76547261e-02 -5.88481277e-02 -7.58901983e-02  2.21602116e-02
  9.90563817

In [27]:
tags = [tag[1:len(tag) - 1].split('><') for tag in df_net['Tags']]
tags

[['python', 'dictionary', 'types', 'typeof'],
 ['python', 'json', 'parsing'],
 ['python', 'string', 'character-encoding', 'python-3.x'],
 ['python', 'integer', 'string-formatting'],
 ['python', 'pandas', 'dataframe', 'append'],
 ['python', 'dictionary'],
 ['python', 'string', 'variables', 'types'],
 ['python', 'exception', 'arguments'],
 ['python', 'properties', 'getter-setter'],
 ['python', 'string', 'integer'],
 ['python', 'dictionary', 'pickle'],
 ['python', 'list', 'python-3.x', 'map-function'],
 ['python', 'pandas', 'group-by', 'pivot', 'pivot-table'],
 ['python', 'variables', 'defined'],
 ['python', 'json'],
 ['python', 'dictionary', 'list-comprehension'],
 ['python', 'lambda', 'conditional-operator'],
 ['python', 'string'],
 ['python', 'windows', 'environment-variables', 'pythonpath'],
 ['python', 'arrays', 'intersection'],
 ['python', 'stdout'],
 ['python', 'subprocess'],
 ['python', 'range'],
 ['python', 'json', 'dictionary', 'serialization'],
 ['python', 'path', 'working-dire

In [28]:
# Utilisation de MultiLabelBinarizer pour créer des étiquettes binaires
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(tags)
labels

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [29]:
# Séparation des données en ensembles d'entraînement et de test
from sklearn.model_selection import train_test_split

train_title, test_title, train_labels, test_labels = train_test_split(embeddings_np, labels, test_size=0.2, random_state=42)
print("train_title shape : {}".format(train_title.shape))
print("test_title shape : {}".format(test_title.shape))
print("train_labels shape : {}".format(train_labels.shape))
print("test_labels shape : {}".format(test_labels.shape))

train_title shape : (40, 512)
test_title shape : (10, 512)
train_labels shape : (40, 66)
test_labels shape : (10, 66)


In [30]:
# Initialize Logistic Regression with OneVsRest
param_logit = {"estimator__C": [100, 10, 1.0, 0.1],
               "estimator__penalty": ["l1", "l2"],
               "estimator__dual": [False],
               "estimator__solver": ["liblinear"]}

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.multiclass import OneVsRestClassifier

lr = GridSearchCV(OneVsRestClassifier(LogisticRegression()),
                              param_grid=param_logit,
                              n_jobs=-1,
                              cv=5,
                              scoring="f1_weighted",
                              return_train_score = True,
                              refit=True)
lr.fit(train_title, train_labels)

# Predict
test_labels_predicted = lr.predict(test_title)

# Inverse transform
test_labels_pred_inversed = mlb\
    .inverse_transform(test_labels_predicted)
test_labels_inversed = mlb\
    .inverse_transform(test_labels)

print("Les 10 premiers tags prédits - vrais tags :")
print("Prédits:", test_labels_pred_inversed[0:10])
print("Vrais:", test_labels_inversed[0:10])

/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 16 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 25 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 33 is present in all training examples.
  warnings.warn(
/usr/local/l

Les 10 premiers tags prédits - vrais tags :
Prédits: [('python',), ('python',), ('python',), ('list', 'python'), ('python',), ('python',), ('list', 'python', 'python-3.x'), ('python',), ('python',), ('list', 'python')]
Vrais: [('defined', 'python', 'variables'), ('beautifulsoup', 'python', 'scrapy', 'ssl-certificate', 'web-scraping'), ('for-loop', 'python'), ('list', 'python'), ('python', 'string'), ('module', 'python', 'python-3.6', 'python-import'), ('list-comprehension', 'python'), ('python',), ('python', 'python-3.x'), ('arrays', 'intersection', 'python')]


/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 63 is present in all training examples.
  warnings.warn(


In [31]:
from sklearn.metrics import coverage_error
from sklearn.metrics import label_ranking_average_precision_score
from sklearn.metrics import label_ranking_loss


def metrics_score(model, df, y_true, y_pred):
    """Compilation function of metrics specific to multi-label
    classification problems in a Pandas DataFrame.
    This dataFrame will have 1 row per metric
    and 1 column per model tested.

    Exemples de Metriques
    ----------------------------------------
    coverage_error = 3 : En moyenne, il faut inclure trois prédictions pour capturer toutes les étiquettes pertinentes pour chaque instance.
    label_ranking_average_precision_score = 0.85 : La précision moyenne des classements est de 85%. Cela signifie que la majorité des étiquettes pertinentes sont bien classées en haut.
    label_ranking_loss = 0.1 : Il y a 10% d'inversions dans les classements où les étiquettes non pertinentes sont classées avant les étiquettes pertinentes.

    Parameters
    ----------------------------------------
    model : string
        Name of the tested model
    df : DataFrame
        DataFrame to extend.
        If None : Create DataFrame.
    y_true : array
        Array of true values to test
    y_pred : array
        Array of predicted values to test
    ----------------------------------------
    """
    if(df is not None):
        df_results = df
    else:
        df_results = pd.DataFrame(index=["coverage_error", "label_ranking_average_precision_score",
                                       "label_ranking_loss"],
                               columns=[model])

    scores = []
    scores.append(coverage_error(y_true,
                                         y_pred))
    scores.append(label_ranking_average_precision_score(y_pred,
                                   y_true))
    scores.append(label_ranking_loss(y_true,
                                       y_pred))

    df_results[model] = scores

    return df_results

In [32]:
import sklearn.metrics as metrics
df_metrics_compare_use = metrics_score("Logit",
                                   df=None,
                                   y_true = test_labels,
                                   y_pred = test_labels_predicted)
df_metrics_compare_use

,Logit
coverage_error,53.100000
label_ranking_average_precision_score,0.466212
label_ranking_loss,0.490425


- coverage_error = 53.1 : Le modèle nécessite de parcourir un grand nombre de prédictions pour couvrir toutes les étiquettes pertinentes (53/66).
- label_ranking_average_precision_score = 0.466212 : La précision moyenne des classements des étiquettes est faible.
- label_ranking_loss = 0.490425 : Il y a 49% d'inversions dans les classements des étiquettes. C'est élevé.

Ce modèle donne des résultats meilleurs que BERT et Word2Vec mais cela reste faible.

In [37]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Initialiser le classificateur RandomForest
rf_model = RandomForestClassifier(random_state=42)
print("rf_model:", rf_model)

# Utiliser MultiOutputClassifier pour le modèle multi-label
rf_multioutput = OneVsRestClassifier(rf_model)
print("rf_multioutput:", rf_multioutput)

# Définir les hyperparamètres pour RandomizedSearchCV
param_rf = {
    "estimator__n_estimators": [50, 100, 200],
    "estimator__max_depth": [5,6,7],
    "estimator__min_samples_split": [2, 5, 10]
}

random_rf = RandomizedSearchCV(rf_multioutput, param_distributions=param_rf, cv=5, scoring='f1_weighted', n_jobs=-1)
random_rf.fit(train_title, train_labels)

# Meilleurs hyperparamètres
print("Meilleurs hyperparamètres trouvés par GridSearchCV:")
print(random_rf.best_params_)

# Prédictions avec le modèle optimisé
pred_labels = random_rf.predict(test_title)

# Inverse transform des prédictions optimisées
test_labels_pred_inversed_optimized = mlb.inverse_transform(pred_labels)
test_labels_inversed = mlb.inverse_transform(test_labels)

# Évaluer les performances avec le modèle optimisé
print("Les 10 premiers tags prédits avec le modèle optimisé - vrais tags :")
print("Prédits:", test_labels_pred_inversed_optimized[0:10])
print("Vrais:", test_labels_inversed[0:10])

df_metrics_compare_use_rf = metrics_score("Random Forest",
                                   df=None,
                                   y_true = test_labels,
                                   y_pred = pred_labels)

df_metrics_compare_use_rf

rf_model: RandomForestClassifier(random_state=42)
rf_multioutput: OneVsRestClassifier(estimator=RandomForestClassifier(random_state=42))


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 16 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 25 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/multiclass.py:77: UserWarning: Label not 33 is present in all training examples.
  warnings.warn(
/usr/local/l

Meilleurs hyperparamètres trouvés par GridSearchCV:
{'estimator__n_estimators': 50, 'estimator__min_samples_split': 5, 'estimator__max_depth': 5}
Les 10 premiers tags prédits avec le modèle optimisé - vrais tags :
Prédits: [('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',)]
Vrais: [('defined', 'python', 'variables'), ('beautifulsoup', 'python', 'scrapy', 'ssl-certificate', 'web-scraping'), ('for-loop', 'python'), ('list', 'python'), ('python', 'string'), ('module', 'python', 'python-3.6', 'python-import'), ('list-comprehension', 'python'), ('python',), ('python', 'python-3.x'), ('arrays', 'intersection', 'python')]


,Random Forest
coverage_error,59.500000
label_ranking_average_precision_score,0.461667
label_ranking_loss,0.538333


In [38]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialiser le classificateur GradientBoostingClassifier
gb_model = GradientBoostingClassifier(random_state=42)
gb_multioutput = OneVsRestClassifier(gb_model, n_jobs=-1)

# Entraînement du modèle
gb_multioutput.fit(train_title, train_labels)
print("Modèle entraîné.")

# Prédictions
pred_labels = gb_multioutput.predict(test_title)

# Inverse transform des prédictions
test_labels_pred_inversed = mlb.inverse_transform(pred_labels)
test_labels_inversed = mlb.inverse_transform(test_labels)

# Évaluer les performances avec le modèle optimisé
print("Les 10 premiers tags prédits avec le modèle optimisé - vrais tags :")
print("Prédits:", test_labels_pred_inversed_optimized[0:10])
print("Vrais:", test_labels_inversed[0:10])


df_metrics_compare_use_gb = metrics_score("Gradient Boosting",
                                   df=None,
                                   y_true = test_labels,
                                   y_pred = pred_labels)
df_metrics_compare_use_gb

Modèle entraîné.
Les 10 premiers tags prédits avec le modèle optimisé - vrais tags :
Prédits: [('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',), ('python',)]
Vrais: [('defined', 'python', 'variables'), ('beautifulsoup', 'python', 'scrapy', 'ssl-certificate', 'web-scraping'), ('for-loop', 'python'), ('list', 'python'), ('python', 'string'), ('module', 'python', 'python-3.6', 'python-import'), ('list-comprehension', 'python'), ('python',), ('python', 'python-3.x'), ('arrays', 'intersection', 'python')]


,Gradient Boosting
coverage_error,53.100000
label_ranking_average_precision_score,0.457879
label_ranking_loss,0.490677


In [39]:
pd.concat([df_metrics_compare_use, df_metrics_compare_use_rf, df_metrics_compare_use_gb], axis=1)

,Logit,Random Forest,Gradient Boosting
coverage_error,53.100000,59.500000,53.100000
label_ranking_average_precision_score,0.466212,0.461667,0.457879
label_ranking_loss,0.490425,0.538333,0.490677


Je décide de travailler avec la Régression Logistique qui me donne les meilleurs résultats.
